# SAC Irrigation Training — v2.7.0 (Kaggle)

## Key changes from v2.5
| # | Change | Effect |
|---|--------|--------|
| 1 | gamma obs slot restored to `elev_norm` | Bug fix: was writing field-uniform GDD scalar instead of per-agent elevation |
| 2 | 3 new static topo features per agent | Actor can now differentiate agents by cascade role, not just soil moisture |
| 3 | OBS_DIM 707 → 1097 | Per-agent block grows from 5 to 8 features |
| 4 | Episodes always run 93 days | Agent now feels late-season drought after overspending; no early termination |
| 5 | Reward simplified to r1+r2+r3+r6 | Burn-rate `rb` and dead delta-u `r5` removed |

## Diagnostic checks to run BEFORE the full 500k job
Run Cell 3 (smoke test + 25k pilot). Expected at step 25k:
- `critic_loss < 50` (was ~2700 at step 1k, converges fast)
- `ep_len_mean == 93` (full-season episodes — was ~83 in v2.5 due to early termination)
- `ent_coef` constant at 0.05

## WandB metrics to watch
| Metric | Expected v2.7 behaviour | Abort if |
|--------|------------------------|----------|
| `train/critic_loss` | Falls to <50 by step 25k | Exceeds 500 after step 50k |
| `rollout/ep_len_mean` | Reaches **exactly 93** and stays | Below 93 at step 25k |
| `rollout/ep_rew_mean` | More negative early than v2.5 (normal); improves over time | Monotonically decreasing after step 100k |

**Change `SEED` to 0, 1, 2, 3, 4 and run one Kaggle notebook per seed.**

**Submit as Save Version → Save & Run All** to avoid browser-disconnect zombie sessions.

In [ ]:
# ── Cell 1: Install & clone ──────────────────────────────────────────────────
import subprocess, sys, os, torch

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

run('pip install stable-baselines3==2.6.0 gymnasium wandb pytest --quiet')

if os.path.exists('/kaggle/working/thesis'):
    run('cd /kaggle/working/thesis && git pull')
else:
    run('git clone https://github.com/taratorbati/thesis.git /kaggle/working/thesis')

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')

import numpy as np, gymnasium, stable_baselines3 as sb3
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA:              {torch.cuda.is_available()}')
torch.set_num_threads(4)
print('Setup complete.')

In [ ]:
# ── Cell 2: WandB secret ────────────────────────────────────────────────────
# Store WANDB_API_KEY in: Add-ons → Secrets → Add new secret
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('✓  WandB API key loaded from Kaggle secrets.')
except Exception as e:
    print(f'⚠  Could not load WandB key ({e}); training will continue without WandB.')

In [ ]:
# ── Cell 3: Smoke test + 25k pilot (READ OUTPUT before running Cell 4) ───────
#
# This cell does three things:
#   3a. Full smoke tests and VDN unit tests (all 19 tests must pass)
#   3b. obs_dim sanity check (must be 1097, not 707)
#   3c. 25k pilot training run — the mandatory stability check before 500k
#
# Abort criteria (do NOT proceed to Cell 4 if any of these fire):
#   - Any test fails in 3a
#   - critic_loss > 500 at step 25k in 3c
#   - ep_len_mean != 93 in 3c

import subprocess, sys

# 3a. Full test suite
print('Running smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/test_rl_smoke.py', 'tests/test_factorized_critic.py',
     '-v', '--tb=short'],
    capture_output=False
)
if r.returncode != 0:
    raise RuntimeError('TESTS FAILED — do not proceed to Cell 4')
print()

# 3b. obs_dim check
from src.rl.gym_env import IrrigationEnv, OBS_DIM
env_check = IrrigationEnv(randomize=False)
obs_check, _ = env_check.reset()
assert obs_check.shape[0] == 1097, f'Wrong obs_dim: {obs_check.shape[0]} (expected 1097)'
print(f'obs_dim: {obs_check.shape[0]} ✓')
print()

# 3c. 25k pilot run — check critic_loss and ep_len_mean on WandB
print('Running 25k pilot...')
from src.rl.train import train_sac
train_sac(
    seed=0,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=25_000,
)
print()
print('✓  Pilot complete. Check WandB before running Cell 4:')
print('   critic_loss < 50 at step 25k  →  proceed')
print('   ep_len_mean == 93              →  proceed')
print('   Either condition fails         →  STOP and investigate')

In [ ]:
# ── Cell 4: Full 500k training ───────────────────────────────────────────────
# Only run after Cell 3 pilot passed both abort criteria.
# Change SEED for each parallel Kaggle session (0, 1, 2, 3, 4).

SEED = 0   # ← CHANGE THIS

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=500_000,
)
print(f'Training complete for seed {SEED}.')

In [ ]:
# ── Cell 5: Copy results to output (replay buffer excluded) ──────────────────
import shutil, os

src = f'/kaggle/working/thesis/results/rl/sac_general_seed{SEED}'
dst = f'/kaggle/working/results_seed{SEED}_v27'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))

files = [f for root, _, files in os.walk(dst) for f in files]
print(f'Results copied to {dst} ({len(files)} files, replay buffer excluded).')
for root, _, flist in os.walk(dst):
    for f in flist:
        path = os.path.join(root, f)
        rel  = os.path.relpath(path, dst)
        print(f'  {rel}  ({os.path.getsize(path)/1e6:.2f} MB)')

In [ ]:
# ── Cell 6: Zip & download ───────────────────────────────────────────────────
import shutil
archive = f'/kaggle/working/sac_seed{SEED}_v27'
shutil.make_archive(archive, 'zip', f'/kaggle/working/results_seed{SEED}_v27')
print(f'Archive ready: {archive}.zip')
print('Download via Kaggle → Output → Files panel.')

In [ ]:
# ── Cell 7: Resume from checkpoint (if session was killed) ───────────────────
# Uncomment and run to resume from the latest checkpoint.
# Upload the checkpoint zip to a Kaggle Dataset first, then add it as Input.

# SEED = 0
# CHECKPOINT_STEPS = 200_000   # match the checkpoint filename
# CHECKPOINT_ZIP = f'/kaggle/input/sac-checkpoint/sac_general_seed{SEED}_{CHECKPOINT_STEPS}_steps.zip'
# BUFFER_PKL     = f'/kaggle/input/sac-checkpoint/replay_buffer_latest.pkl'
#
# from stable_baselines3 import SAC
# from src.rl.train import _make_lr_schedule, LR_START, LR_END
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
# model = SAC.load(CHECKPOINT_ZIP, env=env)
# model.load_replay_buffer(BUFFER_PKL)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
# remaining = 500_000 - CHECKPOINT_STEPS
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'/kaggle/working/thesis/results/rl/sac_general_seed{SEED}/sac_general_seed{SEED}_final')
# print('Resume complete.')